## 1: setup and load

In [1]:
import sys
sys.path.append("..")

import pandas as pd 
import numpy as np

%load_ext autoreload
%autoreload 2
from src.evaluation import evaluate

train = pd.read_csv("../data/processed/train.csv")
val = pd.read_csv("../data/processed/val.csv")
test = pd.read_csv("../data/processed/test.csv")
movies = pd.read_csv("../data/ml-latest-small/movies.csv")

catalog = train["movieId"].unique()
print(len(catalog), "movies in the catalogue")

3637 movies in the catalogue


## 2: baseline 1, random

In [2]:
rng = np.random.default_rng(42)

def recommend_random(user_id, seen, k):
    candidates = [m for m in catalog if m not in seen]
    return list(rng.choice(candidates, size=k, replace=False))

## 3: baseline 2, most popular

In [3]:
popularity_rank = train.groupby("movieId").size().sort_values(ascending=False).index.tolist()

def recommend_popular(user_id, seen, k):
    recs = []
    for movie_id in popularity_rank:
        if movie_id not in seen:
            recs.append(movie_id)
            if len(recs) == k:
                break
    return recs

top10 = movies.set_index("movieId").loc[popularity_rank[:10], "title"]
print(top10.to_string())

movieId
356                           Forrest Gump (1994)
318              Shawshank Redemption, The (1994)
296                           Pulp Fiction (1994)
2571                           Matrix, The (1999)
593              Silence of the Lambs, The (1991)
260     Star Wars: Episode IV - A New Hope (1977)
110                             Braveheart (1995)
480                          Jurassic Park (1993)
589             Terminator 2: Judgment Day (1991)
2959                            Fight Club (1999)


## 4: score both on the validation set

In [4]:
results = {}
for name, fn in [("Random", recommend_random), ("Most popular", recommend_popular)]:
    r = evaluate(fn, history=train, target=val, k=10)
    r["coverage"] = r["n_unique_items_recommended"] / len(catalog)
    results[name] = r

pd.DataFrame(results).T.round(4)

,precision@k,recall@k,ndcg@k,users_evaluated,n_unique_items_recommended,coverage
Random,0.0028,0.0026,0.0034,572.0,2867.0,0.7883
Most popular,0.0545,0.0727,0.0793,572.0,80.0,0.0220


In [7]:
print(pd.DataFrame(results).T.round(4).to_string())

              precision@k  recall@k  ndcg@k  users_evaluated  n_unique_items_recommended  coverage
Random             0.0028    0.0026  0.0034            572.0                      2867.0    0.7883
Most popular       0.0545    0.0727  0.0793            572.0                        80.0    0.0220


#### Findings: baselines (validation set, k=10)

- Random recommendations score almost zero (precision@10 0.0028, recall@10 0.0026, NDCG@10 0.0034), as expected when picking 10 movies from a catalogue of 3,637. Most popular scores about 20 times higher (precision@10 0.0545, recall@10 0.0727, NDCG@10 0.0793), roughly one liked movie in the top 10 for every second user. These are the numbers any personalised model has to beat.
- Its weakness is that it gives nearly everyone the same list, so catalogue coverage is very low (random: X%, most popular: Y%).